In [37]:
import requests
import pandas as pd
import os
import io
import datetime as dt


class NFCIData:
    url_address = "https://www.chicagofed.org/-/media/publications/nfci/nfci-data-series-csv.csv?sc_lang=en&hash=4EFA8ECC816E4025BACB1FF7E7E5727B"

    def __str__(self):
        return "National Confidence Index data"

    def __init__(self):
        pass

    def get_data(self):
        try:
            response = requests.get(self.url_address)
            response.raise_for_status()
            data_df = pd.read_csv(io.BytesIO(response.content))  # pd.DataFrame(response.content)
            data_df = data_df.rename(columns={"Friday_of_Week": "AsOf"})
            data_df["AsOf"] = pd.to_datetime(data_df["AsOf"])
            # data is available on next whensday
            data_df["Date"] = data_df["AsOf"] + (4 - data_df["AsOf"].dt.weekday + 5).apply(
                lambda x: dt.timedelta(days=x)
            )
            data_df = data_df.set_index("Date")
            return data_df
        except requests.exceptions.RequestException as e:
            print(f"Error downloading the file: {e}")
            return False
        except Exception as e:
            print(f"An unexpected error occurred: {e}")
            return False

In [38]:
nfci_data = NFCIData()

In [39]:
data_df = nfci_data.get_data()